In [73]:
import escher
import cobra
import pandas as pd

In [74]:
# Load the models
amac_model = cobra.io.read_sbml_model("model.xml")
ecoli_model = cobra.io.read_sbml_model("/Users/helenscott/Documents/PhD/Segre-lab/GEM-repos/ecoli/iJO1366.xml")

In [75]:
# Save the paths to the maps
amac_map_path = "escher/MIT1002_TCA_escher-map.json"
ecoli_map_path = "/Users/helenscott/Documents/PhD/Segre-lab/GEM-repos/ecoli/iJO1366.TCA Only.json"

In [ ]:
# Test that the Escher maps can be created
amac_map = escher.Builder(
    model = amac_model,
    map_json = amac_map_path,
    reaction_data = {'rxn00256_c0': 910.000,
                     'rxn00974_c0': 910.000,
                     'rxn01388_c0': 0.000,
                     'rxn01387_c0': 0.000,
                     'rxn00199_c0': 0.000,
                     'rxn08094_c0': 0.000,
                     'rxn00285_c0': 1000.000,
                     'rxn00288_c0': 766.316,
                     'rxn00799_c0': 1000.000,
                     'rxn00248_c0': 1000.000},
    
)
amac_map

Builder(reaction_data={'rxn00256_c0': 910.0, 'rxn00974_c0': 910.0, 'rxn01388_c0': 0.0, 'rxn01387_c0': 0.0, 'rx…

In [77]:
# Define media without carbon or nitrogen sources
amac_basal_media = {
    "EX_cpd00058_e0": 1000,  # Cu2+_e0
    "EX_cpd00007_e0": 20,  # O2_e0
    "EX_cpd00971_e0": 1000,  # Na+_e0
    "EX_cpd00063_e0": 1000,  # Ca2+_e0
    "EX_cpd00048_e0": 1000,  # Sulfate_e0
    "EX_cpd10516_e0": 1000,  # fe3_e0
    "EX_cpd00254_e0": 1000,  # Mg_e0
    "EX_cpd00009_e0": 1000,  # Phosphate_e0
    "EX_cpd00205_e0": 1000,  # K+_e0
    "EX_cpd00099_e0": 1000,  # Cl-_e0
    "EX_cpd00030_e0": 1000,  # Mn2+_e0
    "EX_cpd00001_e0": 1000,  # H2O_e0
    "EX_cpd00034_e0": 1000,  # Zn2+_e0
    "EX_cpd00149_e0": 1000,  # Co2+_e0
}
ecoli_basal_media = {
    'EX_co2_e': 1000.0,
    'EX_cobalt2_e': 1000.0,
    'EX_h_e': 1000.0,
    'EX_h2o_e': 1000.0,
    'EX_k_e': 1000.0,
    'EX_cu2_e': 1000.0,
    'EX_mg2_e': 1000.0,
    'EX_mn2_e': 1000.0,
    'EX_mobd_e': 1000.0,
    'EX_na1_e': 1000.0,
    'EX_ca2_e': 1000.0,
    'EX_cbl1_e': 0.01,
    'EX_ni2_e': 1000.0,
    'EX_o2_e': 1000.0,
    'EX_cl_e': 1000.0,
    'EX_pi_e': 1000.0,
    'EX_zn2_e': 1000.0,
    'EX_sel_e': 1000.0,
    'EX_slnt_e': 1000.0,
    'EX_so4_e': 1000.0,
    'EX_tungs_e': 1000.0,
    'EX_fe2_e': 1000.0,
    'EX_fe3_e': 1000.0
 }

In [78]:
# Define additions to the minimal media
c_sources = {"Glucose": {"ModelSEED": "EX_cpd00027_e0", "BiGG": "EX_glc__D_e"},
             "Acetate": {"ModelSEED": "EX_cpd00029_e0", "BiGG": "EX_ac_e"}}
n_sources = {"Ammonia": {"ModelSEED": "EX_cpd00013_e0", "BiGG": "EX_nh4_e"},
             "Nitrate": {"ModelSEED": "EX_cpd00209_e0", "BiGG": "EX_no3_e"}}

In [79]:
# Make list of all reactions to maximize
modelseed_tca_rxns = [
    "rxn00256_c0",
    "rxn00974_c0",
    "rxn01388_c0",
    "rxn01387_c0",
    "rxn00199_c0",
    "rxn00336_c0",
    "rxn00441_c0",
    "rxn02376_c0",
    "rxn01872_c0",
    "rxn08094_c0",
    "rxn00285_c0",
    "rxn00288_c0",
    "rxn10126_c0",
    "rxn00799_c0",
    "rxn00248_c0"
]
bigg_tca_rxns = [
    "ACONTa",
    "ACONTb",
    "ICDHyr",
    "AKGDH",
    "SUCOAS",
    "FRD2",
    "SUCDi",
    "FUM",
    "MOX",
    "MDH2",
    "MDH",
    "MDH3",
    "CS",
    "CITL",
    "MALS",
    "ICL"
]

# Optimize for Biomass (Check Fluxes When Growing)

## For Amac

In [80]:
# Make a list to store results
amac_growth_results = []
# Get every combination of C and N source
for c_name, c_id_dict in c_sources.items():
    for n_name, n_id_dict in n_sources.items():
        # Debugging
        print(f'Testing AMAC with C source: {c_name} ({c_id_dict["ModelSEED"]}) and N source: {n_name} ({n_id_dict["ModelSEED"]})')
        # Make a medium
        amac_medium = amac_basal_media.copy()
        # Add a limiting amount of carbon source
        # But use different bounds for glucose and acetate so the number of carbon atoms is the same
        if c_name == "Glucose":
            amac_medium[c_id_dict["ModelSEED"]] = 10
        else:
            amac_medium[c_id_dict["ModelSEED"]] = 30
        amac_medium[n_id_dict["ModelSEED"]] = 1000
        # Set the medium
        amac_model.medium = amac_medium
        # Optimize for Biomass (Check Fluxes When Growing)
        amac_solution = amac_model.optimize()
        # Get the fluxes through the TCA cycle reactions (skip reactions that aren't in the model)
        rxn_fluxes = {rxn: amac_solution.fluxes[rxn] for rxn in modelseed_tca_rxns if rxn in amac_solution.fluxes}
        # Add the biomass flux to the dict
        rxn_fluxes["bio1_biomass"] = amac_solution.objective_value
        # Store the results
        amac_growth_results.append({
            "C_source": c_name,
            "N_source": n_name,
            "fluxes": rxn_fluxes,
        })
# Convert to DataFrame for easier viewing
amac_growth_df = pd.DataFrame(amac_growth_results)
amac_growth_df

Could not identify an external compartment by name and choosing one with the most boundary reactions. That might be complete nonsense or change suddenly. Consider renaming your compartments using `Model.compartments` to fix this.


Testing AMAC with C source: Glucose (EX_cpd00027_e0) and N source: Ammonia (EX_cpd00013_e0)
Testing AMAC with C source: Glucose (EX_cpd00027_e0) and N source: Nitrate (EX_cpd00209_e0)


Could not identify an external compartment by name and choosing one with the most boundary reactions. That might be complete nonsense or change suddenly. Consider renaming your compartments using `Model.compartments` to fix this.
Could not identify an external compartment by name and choosing one with the most boundary reactions. That might be complete nonsense or change suddenly. Consider renaming your compartments using `Model.compartments` to fix this.
Could not identify an external compartment by name and choosing one with the most boundary reactions. That might be complete nonsense or change suddenly. Consider renaming your compartments using `Model.compartments` to fix this.


Testing AMAC with C source: Acetate (EX_cpd00029_e0) and N source: Ammonia (EX_cpd00013_e0)
Testing AMAC with C source: Acetate (EX_cpd00029_e0) and N source: Nitrate (EX_cpd00209_e0)


,C_source,N_source,fluxes
0,Glucose,Ammonia,"{'rxn00256_c0': 1.441541959429365, 'rxn00974_c..."
1,Glucose,Nitrate,"{'rxn00256_c0': 2.915341013993146, 'rxn00974_c..."
2,Acetate,Ammonia,"{'rxn00256_c0': 21.66115161525884, 'rxn00974_c..."
3,Acetate,Nitrate,"{'rxn00256_c0': 15.000000000000185, 'rxn00974_..."


### Map for Gluocse + Ammonia

In [81]:
escher.Builder(
    model = amac_model,
    map_json = amac_map_path,
    reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Glucose"][amac_growth_df["N_source"] == "Ammonia"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_36415/557993070.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Glucose"][amac_growth_df["N_source"] == "Ammonia"]["fluxes"].item()


Builder(reaction_data={'rxn00256_c0': 1.441541959429365, 'rxn00974_c0': 1.4415419594293652, 'rxn01388_c0': -1.…

### Map for Glucose + Nitrate

In [82]:
escher.Builder(
    model = amac_model,
    map_json = amac_map_path,
    reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Glucose"][amac_growth_df["N_source"] == "Nitrate"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_36415/1879366724.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Glucose"][amac_growth_df["N_source"] == "Nitrate"]["fluxes"].item()


Builder(reaction_data={'rxn00256_c0': 2.915341013993146, 'rxn00974_c0': 2.9153410139931464, 'rxn01388_c0': -2.…

### Map for Acetate and Ammonia

In [83]:
escher.Builder(
    model = amac_model,
    map_json = amac_map_path,
    reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Acetate"][amac_growth_df["N_source"] == "Ammonia"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_36415/3190541284.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Acetate"][amac_growth_df["N_source"] == "Ammonia"]["fluxes"].item()


Builder(reaction_data={'rxn00256_c0': 21.66115161525884, 'rxn00974_c0': 21.661151615258845, 'rxn01388_c0': -21…

### Map for Acetate and Nitrate

In [84]:
escher.Builder(
    model = amac_model,
    map_json = amac_map_path,
    reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Acetate"][amac_growth_df["N_source"] == "Nitrate"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_36415/148143941.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = amac_growth_df[amac_growth_df["C_source"] == "Acetate"][amac_growth_df["N_source"] == "Nitrate"]["fluxes"].item()


Builder(reaction_data={'rxn00256_c0': 15.000000000000185, 'rxn00974_c0': 15.000000000000185, 'rxn01388_c0': -1…

## For E. coli

In [85]:
print(ecoli_model.objective)

Maximize
1.0*BIOMASS_Ec_iJO1366_core_53p95M - 1.0*BIOMASS_Ec_iJO1366_core_53p95M_reverse_5c8b1


In [86]:
# Make a list to store results
ecoli_growth_results = []
# Get every combination of C and N source
for c_name, c_id_dict in c_sources.items():
    for n_name, n_id_dict in n_sources.items():
        # Debugging
        print(f'Testing E. coli with C source: {c_name} ({c_id_dict["BiGG"]}) and N source: {n_name} ({n_id_dict["BiGG"]})')
        # Make a medium
        ecoli_medium = ecoli_basal_media.copy()
        # Add a limiting amount of carbon source
        # But use different bounds for glucose and acetate so the number of carbon atoms is the same
        if c_name == "Glucose":
            ecoli_medium[c_id_dict["BiGG"]] = 10
        else:
            ecoli_medium[c_id_dict["BiGG"]] = 30
        # Add an unlimited amount of nitrogen source
        ecoli_medium[n_id_dict["BiGG"]] = 1000
        # Set the medium
        ecoli_model.medium = ecoli_medium
        # Optimize for Biomass (Check Fluxes When Growing)
        ecoli_solution = ecoli_model.optimize()
        # Get the fluxes through the TCA cycle reactions (skip reactions that aren't in the model)
        rxn_fluxes = {rxn: ecoli_solution.fluxes[rxn] for rxn in bigg_tca_rxns if rxn in ecoli_solution.fluxes}
        # Add the biomass flux to the dict
        rxn_fluxes["BIOMASS_Ec_iJO1366_core_53p95M"] = ecoli_solution.objective_value
        # Store the results
        ecoli_growth_results.append({
            "C_source": c_name,
            "N_source": n_name,
            "fluxes": rxn_fluxes,
        })
# Convert to DataFrame for easier viewing
ecoli_growth_df = pd.DataFrame(ecoli_growth_results)
ecoli_growth_df

Testing E. coli with C source: Glucose (EX_glc__D_e) and N source: Ammonia (EX_nh4_e)
Testing E. coli with C source: Glucose (EX_glc__D_e) and N source: Nitrate (EX_no3_e)
Testing E. coli with C source: Acetate (EX_ac_e) and N source: Ammonia (EX_nh4_e)
Testing E. coli with C source: Acetate (EX_ac_e) and N source: Nitrate (EX_no3_e)


,C_source,N_source,fluxes
0,Glucose,Ammonia,"{'ACONTa': 4.857776644226277, 'ACONTb': 4.8577..."
1,Glucose,Nitrate,"{'ACONTa': 7.452338689463803, 'ACONTb': 7.4523..."
2,Acetate,Ammonia,"{'ACONTa': 19.270035975295297, 'ACONTb': 19.27..."
3,Acetate,Nitrate,"{'ACONTa': 20.72967553157012, 'ACONTb': 20.729..."


### Map for Glucose and Ammonia

In [87]:
escher.Builder(
    model = ecoli_model,
    map_json = ecoli_map_path,
    reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Glucose"][ecoli_growth_df["N_source"] == "Ammonia"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_36415/620635751.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Glucose"][ecoli_growth_df["N_source"] == "Ammonia"]["fluxes"].item()


Builder(reaction_data={'ACONTa': 4.857776644226277, 'ACONTb': 4.857776644226277, 'ICDHyr': 4.857776644226276, …

### Map for Glucose and Nitrate

In [88]:
escher.Builder(
    model = ecoli_model,
    map_json = ecoli_map_path,
    reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Glucose"][ecoli_growth_df["N_source"] == "Nitrate"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_36415/880266042.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Glucose"][ecoli_growth_df["N_source"] == "Nitrate"]["fluxes"].item()


Builder(reaction_data={'ACONTa': 7.452338689463803, 'ACONTb': 7.452338689463802, 'ICDHyr': 7.4523386894638035,…

### Map for Acetate and Ammonia

In [89]:
escher.Builder(
    model = ecoli_model,
    map_json = ecoli_map_path,
    reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Acetate"][ecoli_growth_df["N_source"] == "Ammonia"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_36415/423020520.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Acetate"][ecoli_growth_df["N_source"] == "Ammonia"]["fluxes"].item()


Builder(reaction_data={'ACONTa': 19.270035975295297, 'ACONTb': 19.270035975295293, 'ICDHyr': 13.23570238114427…

### Map for Acetate and Nitrate

In [90]:
escher.Builder(
    model = ecoli_model,
    map_json = ecoli_map_path,
    reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Acetate"][ecoli_growth_df["N_source"] == "Nitrate"]["fluxes"].item()
)

/var/folders/sg/v4dtbt1502lfm_z5_nvpyv0h0000gn/T/ipykernel_36415/1460944637.py:4: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  reaction_data = ecoli_growth_df[ecoli_growth_df["C_source"] == "Acetate"][ecoli_growth_df["N_source"] == "Nitrate"]["fluxes"].item()


Builder(reaction_data={'ACONTa': 20.72967553157012, 'ACONTb': 20.729675531570116, 'ICDHyr': 15.51621626556236,…

# Optimize for TCA Cylce Reactions (Check for Blockages)